In [1]:
import os
import json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. SETUP PATH FOLDER
QUERIES_JSON = os.path.abspath("data/eval/queries.json")
PREDICTIONS_CSV = os.path.abspath("data/results/predictions.csv")
EVAL_FOLDER = os.path.abspath("data/eval")

# Load data ground truth dan hasil prediksi
with open(QUERIES_JSON, 'r', encoding='utf-8') as f:
    queries_data = json.load(f)
df_pred = pd.read_csv(PREDICTIONS_CSV)

# =====================================================================
# 2. EVALUASI PREDIKSI SOLUSI (Classification Metrics)
# =====================================================================
# Ambil label asli (ground truth) dan label hasil prediksi sistem
y_true = [q['true_pasal'] for q in queries_data]
y_pred = df_pred['predicted_solution'].tolist()

# Hitung metrik evaluasi prediksi solusi
pred_accuracy = accuracy_score(y_true, y_pred)
pred_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
pred_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
pred_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

# Simpan hasil metrik prediksi ke dalam DataFrame
df_pred_metrics = pd.DataFrame([{
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Value": [pred_accuracy, pred_precision, pred_recall, pred_f1]
}])

pred_metrics_path = os.path.join(EVAL_FOLDER, "prediction_metrics.csv")
df_pred_metrics.to_csv(pred_metrics_path, index=False)

# =====================================================================
# 3. EVALUASI RETRIEVAL (Hit Rate / Relevansi Top-K)
# =====================================================================
# Mengukur seberapa sering dokumen yang satu rumpun pasal masuk ke Top-5
retrieval_hits = []
for idx, q in enumerate(queries_data):
    # Ambil list top 5 ids dari hasil prediksi (mengubah string list kembali ke list objek)
    top_5_str = df_pred.iloc[idx]['top_5_case_ids']
    top_5_ids = json.loads(top_5_str)
    
    # Cek apakah ada ground truth id yang berhasil terjaring di top 5
    hit = 1 if any(cid in top_5_ids for cid in q['ground_truth_case_ids']) else 0
    retrieval_hits.append(hit)

retrieval_accuracy = sum(retrieval_hits) / len(retrieval_hits)

# Simpan hasil metrik retrieval
df_ret_metrics = pd.DataFrame([{
    "Retrieval_Accuracy_Top5": retrieval_accuracy,
    "Total_Queries_Tested": len(queries_data)
}])

ret_metrics_path = os.path.join(EVAL_FOLDER, "retrieval_metrics.csv")
df_ret_metrics.to_csv(ret_metrics_path, index=False)

print("=======================================================")
print("PERFORMA EVALUASI SISTEM CBR")
print("=======================================================")
print(f"Accuracy Prediksi Solusi : {pred_accuracy:.2f}")
print(f"Precision Prediksi Solusi: {pred_precision:.2f}")
print(f"Recall Prediksi Solusi   : {pred_recall:.2f}")
print(f"F1-Score Prediksi Solusi : {pred_f1:.2f}")
print(f"Retrieval Hit Rate (Top5): {retrieval_accuracy:.2f}")
print("=======================================================")
print(f"File metrik berhasil disimpan di folder: {EVAL_FOLDER}")

PERFORMA EVALUASI SISTEM CBR
Accuracy Prediksi Solusi : 0.60
Precision Prediksi Solusi: 0.50
Recall Prediksi Solusi   : 0.50
F1-Score Prediksi Solusi : 0.44
Retrieval Hit Rate (Top5): 1.00
File metrik berhasil disimpan di folder: d:\Kuliah\Semester 6\Computer Reasoning\Sub-CPMK3\cbr-pembunuhan\notebooks\data\eval
